In [55]:
import torch
import torch.nn as nn

import torchvision.transforms as transforms
from torchvision import datasets
from torch.utils.data import DataLoader, SubsetRandomSampler

import numpy as np
import matplotlib.pyplot as plt

import scienceplots
plt.style.use(['science','notebook','grid'])

import numpy as np
from torchvision.models.feature_extraction import get_graph_node_names, create_feature_extractor

In [56]:
from models.models import LeNet5

In [57]:
def display_collapsed_filters(features, img_num, layers, cmap, name_layers, collapse_func=np.mean):
    """
    Display the collapsed feature maps for each layer of interest.

    :param features: dictionary containing the extracted features for each layer
    :param img_num: index of the image in the batch
    :param layers: list of layer names to display the feature maps
    :param collapse_func: function to use for collapsing channels (default: np.mean)
    """

    num_layers = len(layers)
    fig, axes = plt.subplots(1, num_layers, figsize=(25, 8))

    for count, ax in enumerate(axes):
        collapsed_feature_map = collapse_func(features[layers[count]][img_num], axis=0)
        im = ax.imshow(collapsed_feature_map.squeeze(), cmap=cmap)
        ax.set_title(f'{name_layers[count]}')

    fig.subplots_adjust(right=0.8)
    cbar_ax = fig.add_axes([0.85, 0.15, 0.05, 0.7])
    fig.colorbar(im, cax=cbar_ax)

    # fig.savefig('drive/MyDrive/StructureLoss/VGG16/filters.pdf', bbox_inches='tight')

    plt.show()

In [58]:
my_local_data = '/mnt/g/My Drive/types/'

In [59]:
ToTensorAndNormalize = transforms.Compose(
    [
    transforms.Resize((244,244)),
    # transforms.RandomHorizontalFlip(),
    # transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5],  # mean
                         [0.5, 0.5, 0.5] # std
                         )
    ]
)
aux_data = datasets.ImageFolder(
    root=my_local_data,
    transform=ToTensorAndNormalize
)

In [60]:
class_indices = aux_data.class_to_idx
# Original names and their order which you know
names_dict = {'typeA': 'Diameter\nFluctuations',
              'typeB': 'Node Cut',
              'typeC': 'Particle Hit',
              'goodIngots': 'No Structure\nLoss'
              }

# Get class indices from the ImageFolder
class_indices = aux_data.class_to_idx

# Order your names list according to the class indices
ordered_names = [names_dict[class_name] for class_name, index in sorted(class_indices.items(), key=lambda item: item[1])]


In [61]:
def process_plot_image(data, x: int):
    image_data = np.transpose(data[x][0])

    # Normalize or rescale the image data
    if image_data.dtype == np.float32 or image_data.dtype == np.float64:
        if image_data.min() < 0 or image_data.max() > 1:
            image_data = (image_data - image_data.min()) / (image_data.max() - image_data.min())
    elif image_data.dtype == np.uint8:
        if image_data.min() < 0 or image_data.max() > 255:
            image_data = np.clip(image_data, 0, 255)
    else:
        # Rescale to [0, 1] for other data types
        image_data = (image_data - image_data.min()) / (image_data.max() - image_data.min())

    # Display the image
    plt.imshow(image_data)

In [62]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
model = LeNet5()
model = model.to(device=device)
batch_size = 16

In [65]:
# Determine the layer from which to extract features --> we want to extract at the pooling layers
layers = ["convStack.1", "convStack.3"]
node_names = get_graph_node_names(model)
feature_extractor = create_feature_extractor(model, layers)

In [68]:
loader = DataLoader(
    dataset=aux_data,
    batch_size=batch_size,
    shuffle=True,
    num_workers=8
)

In [ ]:
features = {layer_name: [] for layer_name in layers}
labels = []

feature_extractor.eval()
with torch.no_grad():
    for data in loader:
        image, label = data[0].to(device), data[1].to(device)
        predicted_dict = feature_extractor(image)
        for layer_name in layers:
            features[layer_name].extend(predicted_dict[layer_name].cpu().numpy())
        labels.extend(label.cpu().numpy())